# Working with parquet files

## Objective

+ In this assignment, we will use the data downloaded with the module `data_manager` to create features.

(11 pts total)

## Prerequisites

+ This notebook assumes that price data is available to you in the environment variable `PRICE_DATA`. If you have not done so, then execute the notebook `01_materials/labs/2_data_engineering.ipynb` to create this data set.


+ Load the environment variables using dotenv. (1 pt)

In [1]:
# Write your code below.
%load_ext dotenv
%dotenv 


In [2]:
import dask.dataframe as dd

+ Load the environment variable `PRICE_DATA`.
+ Use [glob](https://docs.python.org/3/library/glob.html) to find the path of all parquet files in the directory `PRICE_DATA`.

(1pt)

In [3]:
%run ../../01_materials/labs/update_path.py


In [ ]:
import os
from glob import glob

# Write your code below.

from utils.logger import get_logger

# Initialize logger
_logs = get_logger(__name__)

# Load environment variable
PRICE_DATA = os.getenv("PRICE_DATA")

# Find all parquet files recursively
parquet_files = glob(os.path.join(PRICE_DATA, "**/*.parquet"), recursive=True)

_logs.info(f'Found {len(parquet_files)} parquet files for reading back into Dask.')


2026-01-19 16:31:11,679, 3120178858.py, 19, INFO, Found 2599 parquet files for reading back into Dask.


For each ticker and using Dask, do the following:

+ Add lags for variables Close and Adj_Close.
+ Add returns based on Close:
    
    - `returns`: (Close / Close_lag_1) - 1

+ Add the following range: 

    - `hi_lo_range`: this is the day's High minus Low.

+ Assign the result to `dd_feat`.

(4 pt)

In [9]:
# Write your code below.
 
dd_feat = (
    dd_px
        .groupby('ticker', group_keys=False)
        .apply(
            lambda x: x.sort_values('Date', ascending=True)
                       .assign(
                           Close_lag_1 = x['Close'].shift(1),
                           Adj_Close_lag_1 = x['Adj Close'].shift(1)
                       ),
            meta=pd.DataFrame(
                data={
                    'Date': 'datetime64[ns]',
                    'Open': 'f8',
                    'High': 'f8',
                    'Low': 'f8',
                    'Close': 'f8',
                    'Adj Close': 'f8',
                    'Volume': 'i8',
                    'source': 'object',
                    'Year': 'int32',
                    'Close_lag_1': 'f8',
                    'Adj_Close_lag_1': 'f8'
                },
                index=pd.Index([], dtype=pd.StringDtype(), name='ticker')
            )
        )
        .assign(
            returns = lambda x: x['Close'] / x['Close_lag_1'] - 1,
            hi_lo_range = lambda x: x['High'] - x['Low']
        )
)

+ Convert the Dask data frame to a pandas data frame. 
+ Add a new feature containing the moving average of `returns` using a window of 10 days. There are several ways to solve this task, a simple one uses `.rolling(10).mean()`.

(3 pt)

In [10]:
# Write your code below.

df_feat = dd_feat.compute()

In [11]:
#Add 10-day moving average of returns

df_feat = (
    df_feat
        .sort_values(['ticker', 'Date'])
        .assign(
            returns_ma_10 = (
                df_feat
                    .groupby('ticker')['returns']
                    .rolling(10)
                    .mean()
                    .reset_index(level=0, drop=True)
            )
        )
)

Please comment:

+ Was it necessary to convert to pandas to calculate the moving average return?
    + It was not necessary to convert to pandas but converting to pandas simplifies the implementation.
+ Would it have been better to do it in Dask? Why?
    + Dask is useful for larger or production scale settings so using Dask is more appropriate for scalability. 

(1 pt)

## Criteria

The [rubric](./assignment_1_rubric_clean.xlsx) contains the criteria for grading.

## Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

### Submission Parameters:
* Submission Due Date: `HH:MM AM/PM - DD/MM/YYYY`
* The branch name for your repo should be: `assignment-1`
* What to submit for this assignment:
    * This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
* What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    * Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

Checklist:
- [ ] Created a branch with the correct naming convention.
- [ ] Ensured that the repository is public.
- [ ] Reviewed the PR description guidelines and adhered to them.
- [ ] Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack at `#cohort-3-help`. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.